# 90 Commissioning (flagship)

KOH first fill, wetting, residual oxygen, formation, **grid power in**, rest, first charge, H2/O2, carbonation risk, temperature. Plot every engineer-relevant signal.

## Sequence (verification-scaled seconds; engineering analog in hours)

| t | Action |
| --- | --- |
| 0-600 s | Dry vessel, residual air O2, fans on |
| 600-2400 s | 6 M KOH make-up fill into anode pores |
| 2400-3600 s | Rest / wetting / residual O2 |
| 3600-7200 s | First charge from grid, OER + HER, T rise |

Chemistry: IA-SYS-020/021. Grid: IA-GRD-001. Same `PlantModel` as decade 40.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


In [ ]:
from plant_sim.plant import PlantModel
from plant_sim.scenarios.mission import commissioning_inputs
plant = PlantModel(P)
res = plant.simulate((0, 7200), inputs=commissioning_inputs(P), y0=plant.y0(filled=False), n_eval=360, rtol=1e-4)
t, a, y = res["t"], res["alg"], res["y"]
b = plant.blocks
timeseries(t/3600, {"P_grid_W": a["P_grid_W"], "I_A": a["I_cell_A"], "V_V": a["V_cell_V"]}, xlabel="t (h)", ylabel="P (W) / I (A) / V (V)", title="Grid power, current, voltage", callout="90_Commissioning", stem="nb90_elec")
timeseries(t/3600, {"n_OH": y[b["ely"][0]], "s_wet": y[b["ely"][0]+6], "V_m3": y[b["ely"][0]+5]}, xlabel="t (h)", ylabel="n_OH / wetting / volume", title="KOH fill and wetting", callout="6 M KOH make-up", stem="nb90_chem")
timeseries(t/3600, {"n_H2": a["n_H2_mol"], "n_O2": a["n_O2_oer_mol"], "CE": a["CE"]}, xlabel="t (h)", ylabel="mol / CE", title="Gas evolution", callout="HER + OER on first charge", stem="nb90_gas")
timeseries(t/3600, {"T_fe": y[b["fe"][0]+3], "T_ely": y[b["ely"][0]+4], "T_ves": y[b["th"][0]+2]}, xlabel="t (h)", ylabel="T (K)", title="Temperature rise", callout="I*eta + formation", stem="nb90_T")
timeseries(t/3600, {"SOC": a["SOC"], "n_CO3": y[b["ely"][0]+2], "pO2_atm": a["p_O2_Pa"]/P.P_atm_Pa, "s_gdl": a["s_gdl"]}, xlabel="t (h)", ylabel="SOC / carbonate / pO2 / s", title="SOC, carbonation risk, pO2, GDL saturation", callout="US CO2; pO2 window", stem="nb90_soc")
print("finite", np.all(np.isfinite(y)), "nfev", res["nfev"])
